# YuvaIntern Week 3 – Unsupervised Learning and Clustering Analysis

**Internship:** Virtual Data Science with Python Trainee  
**Task:** Unsupervised Learning and Clustering Analysis  
**Dataset:** Breast Cancer Wisconsin (Diagnostic) Dataset

This notebook applies K-Means clustering to segment observations without using the diagnosis target during clustering. Standardization is performed before clustering, the number of clusters is evaluated using silhouette scores, and PCA is used only to visualize the resulting clusters in two dimensions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

df = pd.read_csv("YuvaIntern_Week3_Breast_Cancer_Clustering_Dataset.csv")
features = list(df.columns[:30])
X = df[features]

print("Shape:", df.shape)
df.head()


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)


In [ ]:
silhouette_scores = {}

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    silhouette_scores[k] = silhouette_score(X_scaled, labels)

silhouette_scores


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), marker="o")
plt.title("Silhouette Score for Different Numbers of Clusters")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(list(silhouette_scores.keys()))
plt.show()


In [ ]:
best_k = max(silhouette_scores, key=silhouette_scores.get)
print("Selected number of clusters:", best_k)


In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

df["Cluster"].value_counts().sort_index()


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Variance explained by PC1 + PC2:", pca.explained_variance_ratio_.sum())


In [ ]:
plt.figure(figsize=(8,5))
for c in sorted(df["Cluster"].unique()):
    mask = df["Cluster"] == c
    plt.scatter(X_pca[mask,0], X_pca[mask,1], s=18, label=f"Cluster {c}")

plt.title(f"K-Means Clusters Visualized with PCA (k={best_k})")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.show()


In [ ]:
cluster_sizes = df["Cluster"].value_counts().sort_index()
cluster_sizes


In [ ]:
cluster_means = df.groupby("Cluster")[features].mean()
cluster_means[["mean radius", "mean texture", "mean perimeter", "mean area"]]


In [ ]:
# Optional comparison: hierarchical clustering
hier_scores = {}

for k in range(2, 6):
    model = AgglomerativeClustering(n_clusters=k)
    labels = model.fit_predict(X_scaled)
    hier_scores[k] = silhouette_score(X_scaled, labels)

hier_scores


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(list(hier_scores.keys()), list(hier_scores.values()), marker="o")
plt.title("Hierarchical Clustering Silhouette Scores")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.xticks(list(hier_scores.keys()))
plt.show()


## Interpretation

The diagnosis target is not used as an input to K-Means. The algorithm groups observations using similarities among the standardized numerical measurements. Silhouette scores provide a quantitative way to compare candidate values of k, while PCA provides a two-dimensional visual representation of the clusters.

Cluster profiles are examined using group-wise feature means. These profiles help describe how the groups differ in their average measurements. Because clustering is unsupervised, cluster labels such as Cluster 0 or Cluster 1 do not have an inherent medical meaning; interpretation is based on the feature characteristics observed in each group.


## Conclusion

The clustering workflow demonstrates preprocessing, model selection, K-Means segmentation, visualization, and cluster interpretation. The resulting groups can be used as a descriptive segmentation of observations and can support further research-oriented analysis. The notebook also includes hierarchical clustering as a comparison method using silhouette scores.
